In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, average_precision_score

In [18]:
#파일 불러오기
file_path = '/content/drive/MyDrive/itda/itda_sampling_split.csv'

df = pd.read_csv(file_path)
df['date'] = pd.to_datetime(df['date'])

df.head()

,user_id,prod_id,rating,label,date,text,original_review_id,review_id,split,train_mask,valid_mask,test_mask
0,8253,3237,5.0,0,2005-08-13,One of the original pizza places. Worth the lo...,378701,0,train,True,False,False
1,17480,3237,3.0,0,2005-09-19,Expect a wait before your first bite into this...,378700,1,test,False,False,True
2,149466,3237,5.0,0,2005-10-11,Could this pizza be anymore delicious? The wh...,378699,2,train,True,False,False
3,41746,3237,5.0,0,2005-10-27,Shocker! Everyone thinks that this is best pi...,378698,3,train,True,False,False
4,192388,3237,5.0,1,2005-10-28,One of my favorite places in the city. The pi...,374686,4,test,False,False,True


node feature 생성

In [19]:
# text feature
df['text_length'] = df['text'].astype(str).str.len()
df['word_count'] = df['text'].astype(str).str.split().apply(len)

# sampled subgraph 내부 기준 count feature
df['user_review_count'] = df.groupby('user_id')['review_id'].transform('count')
df['prod_review_count'] = df.groupby('prod_id')['review_id'].transform('count')

# date feature
start_date = df['date'].min()
df['days_from_start'] = (df['date'] - start_date).dt.days

# 사용할 feature 목록
feature_cols = [
    'rating',
    'text_length',
    'word_count',
    'user_review_count',
    'prod_review_count',
    'days_from_start',
]

print(df[feature_cols].head())

   rating  text_length  word_count  user_review_count  prod_review_count  \
0     5.0          202          31                  2                990   
1     3.0          303          57                  1                990   
2     5.0          202          32                  1                990   
3     5.0          287          50                  1                990   
4     5.0          393          76                  1                990   

   days_from_start  
0                0  
1               37  
2               59  
3               75  
4               76  


train / valid / test 나누기

In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, average_precision_score

# 랜덤 고정
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# split mask 생성
train_mask = df['split'] == 'train'
valid_mask = df['split'] == 'valid'
test_mask = df['split'] == 'test'

# feature와 label 분리
X = df[feature_cols].values
y = df['label'].values

print("X shape:", X.shape)
print("y shape:", y.shape)

print("train 개수:", train_mask.sum())
print("valid 개수:", valid_mask.sum())
print("test 개수:", test_mask.sum())

X shape: (15000, 6)
y shape: (15000,)
train 개수: 9600
valid 개수: 2400
test 개수: 3000


피처 스케일링

In [21]:
scaler = StandardScaler()

X_train = X[train_mask]
X_valid = X[valid_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_valid = y[valid_mask]
y_test = y[test_mask]

# train data 기준으로만 fit
scaler.fit(X_train)

# valid/test는 transform만
X_train_scaled = scaler.transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

print("X_train_scaled shape:", X_train_scaled.shape)
print("X_valid_scaled shape:", X_valid_scaled.shape)
print("X_test_scaled shape:", X_test_scaled.shape)

X_train_scaled shape: (9600, 6)
X_valid_scaled shape: (2400, 6)
X_test_scaled shape: (3000, 6)


torch tensor로 변환

In [22]:
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_valid_tensor = torch.tensor(X_valid_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_valid_tensor = torch.tensor(y_valid, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

print("X_train_tensor:", X_train_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)

X_train_tensor: torch.Size([9600, 6])
y_train_tensor: torch.Size([9600, 1])


MLP 모델 정의

In [23]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.model(x)

In [24]:
#model, loss, optimizer 설정
input_dim = X_train_tensor.shape[1]

model = MLP(input_dim)

positive_count = y_train.sum()
negative_count = len(y_train) - positive_count

print("positive_count:", positive_count)
print("negative_count:", negative_count)

if positive_count == 0:
    raise ValueError("train set에 label=1인 데이터가 없습니다.")

pos_weight_value = negative_count / positive_count
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("input_dim:", input_dim)
print("pos_weight:", pos_weight_value)

positive_count: 1003
negative_count: 8597
input_dim: 6
pos_weight: 8.571286141575275


평가 함수

In [25]:
def evaluate_model(model, X_tensor, y_true):
    model.eval()

    with torch.no_grad():
        logits = model(X_tensor)
        probs = torch.sigmoid(logits).numpy().reshape(-1)

    preds = (probs >= 0.5).astype(int)

    acc = accuracy_score(y_true, preds)
    macro_f1 = f1_score(y_true, preds, average='macro', zero_division=0)
    pr_auc = average_precision_score(y_true, probs)

    return acc, macro_f1, pr_auc

MLP 학습

In [26]:
EPOCHS = 100

history = []

best_valid_pr_auc = -1
best_model_state = None

for epoch in range(1, EPOCHS + 1):
    model.train()

    optimizer.zero_grad()

    logits = model(X_train_tensor)
    loss = criterion(logits, y_train_tensor)

    loss.backward()
    optimizer.step()

    train_acc, train_f1, train_pr_auc = evaluate_model(model, X_train_tensor, y_train)
    valid_acc, valid_f1, valid_pr_auc = evaluate_model(model, X_valid_tensor, y_valid)

    history.append({
        'epoch': epoch,
        'loss': loss.item(),
        'train_acc': train_acc,
        'train_macro_f1': train_f1,
        'train_pr_auc': train_pr_auc,
        'valid_acc': valid_acc,
        'valid_macro_f1': valid_f1,
        'valid_pr_auc': valid_pr_auc
    })

    if valid_pr_auc > best_valid_pr_auc:
        best_valid_pr_auc = valid_pr_auc
        best_model_state = copy.deepcopy(model.state_dict())

    if epoch % 10 == 0:
        print("epoch:", epoch,
              "loss:", round(loss.item(), 4),
              "valid_macro_f1:", round(valid_f1, 4),
              "valid_pr_auc:", round(valid_pr_auc, 4))

epoch: 10 loss: 1.2447 valid_macro_f1: 0.4762 valid_pr_auc: 0.116
epoch: 20 loss: 1.226 valid_macro_f1: 0.5139 valid_pr_auc: 0.161
epoch: 30 loss: 1.2119 valid_macro_f1: 0.5206 valid_pr_auc: 0.1792
epoch: 40 loss: 1.1884 valid_macro_f1: 0.5073 valid_pr_auc: 0.188
epoch: 50 loss: 1.1763 valid_macro_f1: 0.5052 valid_pr_auc: 0.1966
epoch: 60 loss: 1.1615 valid_macro_f1: 0.5062 valid_pr_auc: 0.2025
epoch: 70 loss: 1.1423 valid_macro_f1: 0.5081 valid_pr_auc: 0.2119
epoch: 80 loss: 1.1411 valid_macro_f1: 0.5145 valid_pr_auc: 0.219
epoch: 90 loss: 1.1297 valid_macro_f1: 0.5174 valid_pr_auc: 0.2223
epoch: 100 loss: 1.1291 valid_macro_f1: 0.5195 valid_pr_auc: 0.2241


best model로 test 성능 확인

In [27]:
# valid PR-AUC가 가장 좋았던 모델 불러오기
model.load_state_dict(best_model_state)

test_acc, test_f1, test_pr_auc = evaluate_model(model, X_test_tensor, y_test)

print("MLP Test Accuracy:", test_acc)
print("MLP Test Macro F1:", test_f1)
print("MLP Test PR-AUC:", test_pr_auc)

MLP Test Accuracy: 0.6136666666666667
MLP Test Macro F1: 0.5153871209493116
MLP Test PR-AUC: 0.25139597572686095


In [28]:
from sklearn.metrics import confusion_matrix, classification_report

model.eval()

with torch.no_grad():
    logits = model(X_test_tensor)
    probs = torch.sigmoid(logits).numpy().reshape(-1)

preds = (probs >= 0.5).astype(int)

print("Confusion Matrix")
print(confusion_matrix(y_test, preds))

print("\nClassification Report")
print(classification_report(y_test, preds, zero_division=0))

Confusion Matrix
[[1596 1090]
 [  69  245]]

Classification Report
              precision    recall  f1-score   support

           0       0.96      0.59      0.73      2686
           1       0.18      0.78      0.30       314

    accuracy                           0.61      3000
   macro avg       0.57      0.69      0.52      3000
weighted avg       0.88      0.61      0.69      3000



best threshold로 다시 학습하자

In [29]:
from sklearn.metrics import f1_score, accuracy_score, average_precision_score

# valid set probability 구하기
model.eval()

with torch.no_grad():
    valid_logits = model(X_valid_tensor)
    valid_probs = torch.sigmoid(valid_logits).numpy().reshape(-1)

best_threshold = 0
best_macro_f1 = -1

for threshold in np.arange(0.05, 0.95, 0.01):
    valid_preds = (valid_probs >= threshold).astype(int)
    macro_f1 = f1_score(y_valid, valid_preds, average='macro', zero_division=0)

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_threshold = threshold

print("Best threshold:", best_threshold)
print("Best valid macro f1:", best_macro_f1)

Best threshold: 0.6700000000000002
Best valid macro f1: 0.5866680744729524


In [30]:
# test set probability
model.eval()

with torch.no_grad():
    test_logits = model(X_test_tensor)
    test_probs = torch.sigmoid(test_logits).numpy().reshape(-1)

test_preds = (test_probs >= best_threshold).astype(int)

test_acc = accuracy_score(y_test, test_preds)
test_macro_f1 = f1_score(y_test, test_preds, average='macro', zero_division=0)
test_pr_auc = average_precision_score(y_test, test_probs)

print("Threshold-tuned MLP Test Accuracy:", test_acc)
print("Threshold-tuned MLP Test Macro F1:", test_macro_f1)
print("Threshold-tuned MLP Test PR-AUC:", test_pr_auc)

print("\nConfusion Matrix")
print(confusion_matrix(y_test, test_preds))

print("\nClassification Report")
print(classification_report(y_test, test_preds, zero_division=0))

Threshold-tuned MLP Test Accuracy: 0.8526666666666667
Threshold-tuned MLP Test Macro F1: 0.6091465600181102
Threshold-tuned MLP Test PR-AUC: 0.25139597572686095

Confusion Matrix
[[2463  223]
 [ 219   95]]

Classification Report
              precision    recall  f1-score   support

           0       0.92      0.92      0.92      2686
           1       0.30      0.30      0.30       314

    accuracy                           0.85      3000
   macro avg       0.61      0.61      0.61      3000
weighted avg       0.85      0.85      0.85      3000



저장

In [31]:
import os
import json

save_dir = '/content/drive/MyDrive/itda'
os.makedirs(save_dir, exist_ok=True)

# 1. node feature 포함 데이터 저장
node_feature_path = save_dir + '/itda_node_features.csv'
df.to_csv(node_feature_path, index=False, encoding='utf-8-sig')

print("저장 완료:", node_feature_path)

저장 완료: /content/drive/MyDrive/itda/itda_node_features.csv


In [32]:
# 2. MLP history 저장
mlp_history_path = save_dir + '/itda_mlp_history.csv'
history_df.to_csv(mlp_history_path, index=False, encoding='utf-8-sig')

print("저장 완료:", mlp_history_path)

NameError: name 'history_df' is not defined

In [33]:
# 3. MLP 최종 결과 저장
mlp_result = pd.DataFrame({
    'model': ['MLP'],
    'test_accuracy': [test_acc],
    'test_macro_f1': [test_macro_f1],
    'test_pr_auc': [test_pr_auc],
    'best_threshold': [best_threshold],
    'note': ['threshold tuned on valid macro f1']
})

mlp_result_path = save_dir + '/itda_mlp_result.csv'
mlp_result.to_csv(mlp_result_path, index=False, encoding='utf-8-sig')

print("저장 완료:", mlp_result_path)

mlp_result

저장 완료: /content/drive/MyDrive/itda/itda_mlp_result.csv


,model,test_accuracy,test_macro_f1,test_pr_auc,best_threshold,note
0,MLP,0.852667,0.609147,0.251396,0.67,threshold tuned on valid macro f1


In [34]:
# 4. 실험 설정 저장
config = {
    'sample_size': 15000,
    'random_state': 42,
    'feature_cols': feature_cols,
    'split_ratio': {
        'train': 0.64,
        'valid': 0.16,
        'test': 0.20
    },
    'mlp_best_threshold': float(best_threshold)
}

config_path = save_dir + '/itda_experiment_config.json'

with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=4)

print("저장 완료:", config_path)

저장 완료: /content/drive/MyDrive/itda/itda_experiment_config.json
